# Evaluate
データセットでエージェントを性能評価。  
定時ジョブなどで実行する事でエージェントの性能を定期的に計測する用途を想定。

# Prepare

In [ ]:
import mlflow
import zoneinfo

tz_info = zoneinfo.ZoneInfo("Asia/Tokyo")
mlflow.set_experiment("agent-rag")

# Define Criteria

In [ ]:
# テストデータを作成
eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "東京都の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "東京都の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "横浜市の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "横浜市の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "群馬県の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "群馬県の天気は豪雨です。"},
    },
]

# Package & Evaluate

In [ ]:
import datetime
from agent_assistant import agents, evaluate
from mlflow.pyfunc.model import ChatAgent

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=f"pkg_{ymd}"):
    # モデルを記録
    res = mlflow.pyfunc.log_model(
        python_model="../src/agent_assistant/pack.py", code_paths=["../src/"], name=f"{agents.AGENT_NAME}_{ymd}"
    )
    logged_model: ChatAgent = mlflow.pyfunc.load_model(res.model_uri) # pyright: ignore[reportAssignmentType]

    # モデルを評価
    mlflow.set_active_model(model_id=res.model_id)
    res = evaluate.eval_responses_cc(logged_model, eval_dataset)
    print(res)